In [ ]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "data/microsoft_2026.pdf" 
COMPANY = "Microsoft"
YEAR = 2026

CHUNK_SIZE = 900
CHUNK_OVERLAP = 120

In [ ]:
def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_no, page in enumerate(doc, start=1):
        pages.append({"page": page_no, "text": page.get_text()})
    return pages

pages = extract_pages(PDF_PATH)
print("Total pages:", len(pages))
print(pages[5]["text"][:600])

In [3]:
empty = [p["page"] for p in pages if len(p["text"].strip()) < 50]
print("Nearly empty pages:", empty)

Nearly empty pages: []


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

def make_chunks(pages, company, year):
    chunks = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            if len(piece.strip()) < 30:      # skip junk like lone page numbers
                continue
            chunks.append({
                "id": f"{company}-{year}-p{p['page']}-{i}",
                "text": piece,
                "metadata": {"company": company, "year": year, "page": p["page"]},
            })
    return chunks

chunks = make_chunks(pages, COMPANY, YEAR)
print("Total chunks:", len(chunks))
print(chunks[50])

In [5]:
lengths = [len(c["text"]) for c in chunks]
print("min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths) // len(lengths))

min: 54 max: 900 avg: 759


In [ ]:
for c in chunks[200:204]:
    print(c["metadata"], "\n", c["text"], "\n" + "-" * 60)